# Credit risk analysis

In [23]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [24]:
df = pd.read_csv("Task 3 and 4_Loan_Data.csv")
df.head()

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,8153374,0,5221.545193,3915.471226,78039.38546,5,605,0
1,7442532,5,1958.928726,8228.752520,26648.43525,2,572,1
2,2256073,0,3363.009259,2027.830850,65866.71246,4,602,0
3,4885975,0,4766.648001,2501.730397,74356.88347,5,612,0
4,4700614,1,1345.827718,1768.826187,23448.32631,6,631,0


In [25]:
# Features and Target
X = df.drop(columns=["customer_id", "default"])
y = df["default"]

In [26]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [27]:
# Logistic Regression Model
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)


Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier', LogisticRegression(max_iter=1000))])

In [28]:
# Model Evaluation
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:,1]

print("\nAccuracy:", accuracy_score(y_test, predictions))
print("ROC AUC :", roc_auc_score(y_test, probabilities))

print("\nClassification Report")
print(classification_report(y_test, predictions))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, predictions))


Accuracy: 0.998
ROC AUC : 0.9999900514010943

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1630
           1       1.00      0.99      0.99       370

    accuracy                           1.00      2000
   macro avg       1.00      0.99      1.00      2000
weighted avg       1.00      1.00      1.00      2000


Confusion Matrix
[[1630    0]
 [   4  366]]


In [29]:
# Function to Calculate Expected Loss
RECOVERY_RATE = 0.10

def expected_loss(
        credit_lines_outstanding,
        loan_amt_outstanding,
        total_debt_outstanding,
        income,
        years_employed,
        fico_score):

    borrower = pd.DataFrame({
        "credit_lines_outstanding":[credit_lines_outstanding],
        "loan_amt_outstanding":[loan_amt_outstanding],
        "total_debt_outstanding":[total_debt_outstanding],
        "income":[income],
        "years_employed":[years_employed],
        "fico_score":[fico_score]
    })

    # Probability of Default
    PD = model.predict_proba(borrower)[0][1]

    # Loss Given Default
    LGD = 1 - RECOVERY_RATE

    # Exposure at Default
    EAD = loan_amt_outstanding

    # Expected Loss
    EL = PD * LGD * EAD

    return {
        "Probability of Default": round(PD,4),
        "Recovery Rate": RECOVERY_RATE,
        "Loss Given Default": LGD,
        "Expected Loss": round(EL,2)
    }

In [30]:
# Example
result = expected_loss(
    credit_lines_outstanding=2,
    loan_amt_outstanding=25000,
    total_debt_outstanding=40000,
    income=65000,
    years_employed=6,
    fico_score=690
)

print("\nExpected Loss Result")
print(result)


Expected Loss Result
{'Probability of Default': 0.9996, 'Recovery Rate': 0.1, 'Loss Given Default': 0.9, 'Expected Loss': 22491.91}
